In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import librosa
import json
import numpy as np
from torch.nn import Linear
from evaluate import load


from transformers import (
    Wav2Vec2Processor, 
    Wav2Vec2FeatureExtractor, 
    Wav2Vec2ForCTC,
    PreTrainedTokenizer,
    Wav2Vec2CTCTokenizer,
    AdamW
    )

from torch.nn.utils.rnn import pad_sequence

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [3]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-large-960h-lv60")

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file="./phonemes_vocab.json", 
    unk_token="<unk>", 
    pad_token="<pad>",
    bos_token="<s>",
    eos_token="</s>",
    word_delimiter_token="|",
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [34]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [35]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-960h-lv60", 
    vocab_size=len(tokenizer.get_vocab()),
    ignore_mismatched_sizes=True)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60 and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60 and are newly initialized because the shapes did not match:
- lm_head.weight: found shape torch.Size([32, 1024]) in the checkpoint and torch.Size([51, 1024]) in the model instantiated
- lm_head.bias: found shape torch.Size([32]) in the checkpoint and torch.Size([51]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
model.state_dict = torch.load("20_epoch.pth")
model.to(device);

In [9]:
df = pd.read_csv("prepared_dataset.csv", delimiter="№")

In [12]:
def make_path(url):
    return "audios/" + url.split("/")[-1]

df["path"] = df["url"].apply(make_path)

In [60]:

def show_result(model, df, word, processor):
    
    df_word = df[df["word"] == word]
    
    transcription = list(df_word["transcription"])[0]
    audio_path = list(df_word["path"])[0]
    
    
    audio, sr = librosa.load(audio_path, sr=16000)
        
    if len(audio.shape) > 2:
        audio = np.mean(audio, axis=1)
    
    audio_inputs = processor(
        audio=audio,
        sampling_rate=sr,
        max_length=16000*10,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).input_values.to(device)
    
    labels = processor(text=transcription).input_ids
        
    inputs = {
    "input_values": audio_inputs,
    "labels": torch.tensor(labels, dtype=torch.int)
    }
    
    outputs = model(**inputs).logits
    predicted_ids = torch.argmax(outputs, dim=-1)
    predicted_transcription = processor.batch_decode(predicted_ids)[0]
    
    print(f"Truth:\t\t / {transcription} /")
    print(f"Predicted:\t / {predicted_transcription} /")
    print("\n")

In [61]:
show_result(model, df, "about", processor)
show_result(model, df, "december", processor)
show_result(model, df, "accept", processor)
show_result(model, df, "zoo", processor)
show_result(model, df, "write", processor)
show_result(model, df, "upon", processor)

Truth:		 / əˈbaʊt /
Predicted:	 / ɔɪ<s>m<s>ɪə<s>ʊəb<s>b<s>jʊəɔɪɑː<s>ɔɪ<s>ɔɪ<s>ɔɪ /


Truth:		 / dɪˈsɛmbə /
Predicted:	 / ɔɪ<s>dj<s>aɪnʊə ʊ<s>ɛɔɪ<s>ɔɪ<s>ɪəˈɔɪpm əʊ<s>ɔɪb<s>bɔɪ /


Truth:		 / əkˈsɛpt /
Predicted:	 / ɔɪ<s>ɔɪ<s>w<s>ɛ<s>ɔɪ<s>ɔɪ<s>ʌ<s>jəʊ<s>ɔɪ<s>ɔɪ /


Truth:		 / zuː /
Predicted:	 / ɔɪbɔɪ<s>ɔɪ<s>ɔɪbʊʌðʊəɛzʊəɛʊeɪ ʊəəʊ<s>ɔɪbɔɪbɔɪ /


Truth:		 / raɪt /
Predicted:	 / ɔɪəʊp<s>bəʊðrʊəɔːuːbm ɑːəʊ<s>ɔɪb<s>ɔɪ /


Truth:		 / əˈpɒn /
Predicted:	 / ɔɪ<s>ˈmhʊ<s>hɔɪɔː<s>ʊəjðeɪ ʊəɑːəʊ<s>ɔɪb<s>ɔɪ /


